## Project 4 — Deepfake Detection 🍎 ↔️ 🍊

Welcome back! In the previous notebook you trained a latent diffusion model that turns apples into oranges (and vice versa) and exported a zip of 100 freshly-minted deepfake fruits. Now we flip roles: instead of being the forger, we become the detective. 🕵️‍♂️

### Outline

1. **Grab the data** — git-lfs-pull the preprocessed fruits from the repo, upload your deepfake zip, and peek at real vs fake side-by-side.
2. **Train / Val / Test splits** — slice the reals 80/20, split the 100 fakes into 60/15/25.
3. **Stage 1 — Learn the real manifold** — train a convolutional autoencoder on real images only (no fakes during training).
4. **Stage 2 — Linear probe on frozen features** — discard the decoder, freeze the encoder, train a single linear layer on real + fake embeddings.
5. **Evaluate the detector** — ROC / PR curves, confusion matrix, and a gallery of correct catches, misses, and false alarms.



### Why a two-stage recipe?

In the real world, labeled fakes are **scarce and expensive** — you'll almost always have orders of magnitude more real images than forged ones. On top of that, generative models get **noticeably better every month**, so any detector you ship today is already drifting out of distribution by next quarter. Training a heavy end-to-end classifier from scratch on a handful of fakes is a losing battle on both fronts.

The classic fix is to split the problem in two:

- **Stage 1 — Pretrain once, on real data only.** A convolutional autoencoder reconstructs real fruit images, and its encoder learns a compressed representation of the real-image manifold. No fakes are needed here, so we can use as much real data as we have.
- **Stage 2 — Retrain cheaply, as often as needed.** Freeze the encoder, discard the decoder, and train a single linear layer (a *linear probe*) on a small balanced mix of real + fake embeddings. When a new generator shows up, you don't touch the encoder — you just grab a handful of fresh fakes and retrain the tiny head in seconds.

This is the same pattern that powers MAE-style pretraining and foundation-model evaluation: expensive self-supervised features, cheap supervised heads on top.


        Input Image
             │
             ▼
     ┌────────────────┐
     │                │
     │   Encoder      │   (shrinking spatial dims,
     │  (Conv AE)     │    increasing channels / compression)
     │                │
     └──────┬─────────┘
            │
            ▼
      Embedding Vector
        (latent z)
            │
            ▼
     ┌──────────────┐
     │  Linear Head │   (frozen encoder, train only this)
     │   (Probe)    │
     └──────┬───────┘
            │
            ▼
        Prediction
     (Real vs Fake)



### Step 1 — Grab the data 🧺

We start by git-lfs-pulling the preprocessed fruits from the repo (same `fruits_train.pt` / `fruits_test.pt` files the diffusion notebook used), uploading the `deepfake_imgs.zip` you generated at the end of that notebook, and previewing real vs fake side-by-side. Everything lives at **128×128**, matching the output of the latent diffusion pipeline — no resizing tricks needed.


In [ ]:
#@title Fetch repository and pull LFS files (this might take around 5-7 minutes) { display-mode: "form" }
# === Clone repo and pull LFS files ===
!git lfs install
!git clone https://github.com/eth-bmai-fs26/project.git
%cd project
!git checkout week4/deepfake
!git lfs pull
%cd /content

DATASET_DIR = 'project/week4/deepfake/dataset'

import os
for f in ['fruits_train.pt', 'fruits_test.pt']:
    size = os.path.getsize(f'{DATASET_DIR}/{f}') / (1024**2)
    print(f"{f}: {size:.0f} MB")


In [ ]:
#@title Install dependencies and imports { display-mode: "form" }
# Deepfake Detection — Frozen Encoder + Linear Probe

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
)
import os, shutil, random, zipfile

# Local override: the latent diffusion pipeline emits 128x128 images, so we
# run the whole detector at 128x128 (utils.py still defines IMG_SIZE=64 for
# the older pixel-space tutorials — we deliberately ignore it here).
IMG_SIZE = 128
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# Constants
BATCH_SIZE = 256
N_FAKES = 100
N_FAKE_TRAIN = 60
N_FAKE_VAL = 15
N_FAKE_TEST = 25

# Autoencoder pretraining
AE_EPOCHS = 200
AE_LR = 1e-3

# Linear probe
PROBE_EPOCHS = 50
PROBE_LR = 1e-3
LATENT_DIM = 128
ERROR_GRID = 8   # side length of the per-image reconstruction-error grid (8x8 = 64 features)


In [ ]:
#@title Load real fruit images from the LFS dataset { display-mode: "form" }
# === Load real images directly from the preprocessed .pt files ===
# These are the same tensors the diffusion notebook trained on, already at 128x128.

train_data = torch.load(f'{DATASET_DIR}/fruits_train.pt', weights_only=False)
test_data  = torch.load(f'{DATASET_DIR}/fruits_test.pt',  weights_only=False)

real_train_all = train_data['composites'].float()   # (N, 3, 128, 128) in [0, 1]
real_test_all  = test_data['composites'].float()
CLASS_NAMES = {0: 'apple', 1: 'orange'}

print(f"Real train: {tuple(real_train_all.shape)}")
print(f"Real test:  {tuple(real_test_all.shape)}")

del train_data, test_data


In [ ]:
#@title Upload and unzip deepfake_imgs.zip { display-mode: "form" }
# === Upload and unzip deepfake images ===
# Generated by the last cell of latent_attention_diffusion_pipeline.ipynb

FAKE_DIR = 'deepfake_imgs'

if not os.path.exists(FAKE_DIR):
    try:
        from google.colab import files
        print("Upload deepfake_imgs.zip:")
        uploaded = files.upload()
        zip_name = list(uploaded.keys())[0]
    except ImportError:
        zip_name = 'deepfake_imgs.zip'
        print(f"Not in Colab — looking for {zip_name} locally")

    with zipfile.ZipFile(zip_name, 'r') as zf:
        zf.extractall('.')
    print(f"Extracted to {FAKE_DIR}/")
else:
    print(f"{FAKE_DIR}/ already exists, skipping upload")

# Load fake images as tensors.
# IMG_SIZE is 128 to match the latent pipeline's output — this Resize is a
# safety net in case you bring in fakes at a different resolution.
fake_files = sorted([f for f in os.listdir(FAKE_DIR) if f.endswith('.png')])
print(f"Found {len(fake_files)} fake images")

to_tensor = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

fake_images = []
for fname in fake_files:
    img = Image.open(os.path.join(FAKE_DIR, fname)).convert('RGB')
    fake_images.append(to_tensor(img))
fake_images = torch.stack(fake_images)

print(f"Fake images tensor: {tuple(fake_images.shape)}")


In [ ]:
#@title Preview real vs fake samples { display-mode: "form" }
# === Visualize real vs fake samples ===

n_show = 8
real_sample = real_train_all[torch.randperm(len(real_train_all))[:n_show]]

fig, axes = plt.subplots(2, n_show, figsize=(2.5 * n_show, 5))
for i in range(n_show):
    axes[0, i].imshow(real_sample[i].permute(1, 2, 0).numpy())
    axes[0, i].axis('off')
    axes[1, i].imshow(fake_images[i].permute(1, 2, 0).clamp(0, 1).numpy())
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Real', fontsize=13, rotation=0, labelpad=40, va='center')
axes[1, 0].set_ylabel('Fake', fontsize=13, rotation=0, labelpad=40, va='center')
plt.suptitle('Real vs Generated (Fake) Samples', fontsize=14)
plt.tight_layout()
plt.show()


### Step 2 — Train / Val / Test splits ✂️

Standard bookkeeping: split the real images 80/20 into train/val, reuse the full real test set as-is, and divide the 100 fakes into 60 train / 15 val / 25 test. The probe training set is deliberately small and balanced — we want to see how well the frozen encoder's features generalize from a handful of examples.


In [ ]:
# === Create train / val / test splits ===
# real_train_all and real_test_all were loaded directly from the .pt files above.

# Split real training images into train and val (80/20)
n_real_total = len(real_train_all)
n_real_val = int(n_real_total * 0.2)

indices = torch.randperm(n_real_total)
real_train = real_train_all[indices[:n_real_total - n_real_val]]
real_val = real_train_all[indices[n_real_total - n_real_val:]]

# Split fake images
fake_train = fake_images[:N_FAKE_TRAIN]
fake_val = fake_images[N_FAKE_TRAIN:N_FAKE_TRAIN + N_FAKE_VAL]
fake_test = fake_images[N_FAKE_TRAIN + N_FAKE_VAL:]

print(f'Train: {len(real_train)} real + {len(fake_train)} fake')
print(f'Val:   {len(real_val)} real + {len(fake_val)} fake')
print(f'Test:  {len(real_test_all)} real + {len(fake_test)} fake')


### Step 3 — Stage 1: Learn the Real Manifold 🧠

We train a convolutional autoencoder on **real images only** — no fakes during training. The encoder compresses each 128×128×3 image into a 128-dim latent, and the decoder tries to reconstruct the original from that bottleneck. Think of this as the same idea as a Masked Autoencoder: self-supervised reconstruction as a proxy task that forces the network to learn the structure of real images.

The architecture is a stack of 5 stride-2 convs (128 → 64 → 32 → 16 → 8 → 4), mirrored on the decoder side. The final 4×4×256 = 4096 feature map gets projected down to a 128-dim latent.

**Heads up for Step 4:** we're not going to use the 128-dim embedding as our detector feature. Instead we're going to exploit the parts of each image the AE **can't** reconstruct — so the cleaner the AE reconstructs real images in this step, the sharper the detector will be later.


In [ ]:
# === Convolutional Autoencoder ===

class ConvAutoencoder(nn.Module):
    """
    Convolutional autoencoder that learns to reconstruct real fruit images.
    The bottleneck forces a compressed representation — the learned manifold.
    Images that don't lie on this manifold (fakes) will reconstruct poorly.
    """
    def __init__(self, latent_dim):
        super().__init__()

        # Encoder: 128x128x3 -> latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1),    # -> 64x64x32
            nn.GroupNorm(8, 32),
            nn.GELU(),

            nn.Conv2d(32, 64, 4, stride=2, padding=1),   # -> 32x32x64
            nn.GroupNorm(8, 64),
            nn.GELU(),

            nn.Conv2d(64, 128, 4, stride=2, padding=1),  # -> 16x16x128
            nn.GroupNorm(8, 128),
            nn.GELU(),

            nn.Conv2d(128, 256, 4, stride=2, padding=1), # -> 8x8x256
            nn.GroupNorm(8, 256),
            nn.GELU(),

            nn.Conv2d(256, 256, 4, stride=2, padding=1), # -> 4x4x256
            nn.GroupNorm(8, 256),
            nn.GELU(),

            nn.Flatten(),                                  # -> 4096
            nn.Linear(256 * 4 * 4, latent_dim),           # -> latent_dim
        )

        # Decoder: latent_dim -> 128x128x3
        self.decoder_fc = nn.Linear(latent_dim, 256 * 4 * 4)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 256, 4, stride=2, padding=1),  # -> 8x8
            nn.GroupNorm(8, 256),
            nn.GELU(),

            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),  # -> 16x16
            nn.GroupNorm(8, 128),
            nn.GELU(),

            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),   # -> 32x32
            nn.GroupNorm(8, 64),
            nn.GELU(),

            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),    # -> 64x64
            nn.GroupNorm(8, 32),
            nn.GELU(),

            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),     # -> 128x128
            nn.Sigmoid(),  # output in [0, 1] to match image range
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        x = self.decoder_fc(z)
        x = x.view(-1, 256, 4, 4)
        return self.decoder(x)

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z)


autoencoder = ConvAutoencoder(latent_dim=LATENT_DIM).to(device)
print(f"Autoencoder parameters: {sum(p.numel() for p in autoencoder.parameters()):,}")


In [ ]:
# === Train autoencoder on REAL images only ===
# This is the key: the model learns what real images look like.
# It never sees a single fake during training.

ae_optimizer = optim.Adam(autoencoder.parameters(), lr=AE_LR, weight_decay=1e-5)
ae_scheduler = optim.lr_scheduler.CosineAnnealingLR(ae_optimizer, T_max=AE_EPOCHS)

ae_train_losses = []
ae_val_losses = []

# Use the real-only train/val splits from Step 2
real_train_loader = DataLoader(
    torch.utils.data.TensorDataset(real_train),
    batch_size=BATCH_SIZE, shuffle=True
)
real_val_loader = DataLoader(
    torch.utils.data.TensorDataset(real_val),
    batch_size=BATCH_SIZE, shuffle=False
)

for epoch in range(AE_EPOCHS):
    autoencoder.train()
    epoch_loss = 0.0
    n_batches = 0
    for (images,) in real_train_loader:
        images = images.to(device)
        recon = autoencoder(images)
        loss = F.mse_loss(recon, images)

        ae_optimizer.zero_grad()
        loss.backward()
        ae_optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    ae_scheduler.step()
    avg_train = epoch_loss / n_batches
    ae_train_losses.append(avg_train)

    # Validation
    autoencoder.eval()
    val_loss = 0.0
    n_val = 0
    with torch.no_grad():
        for (images,) in real_val_loader:
            images = images.to(device)
            val_loss += F.mse_loss(autoencoder(images), images).item()
            n_val += 1
    avg_val = val_loss / n_val
    ae_val_losses.append(avg_val)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{AE_EPOCHS}  train_mse={avg_train:.6f}  val_mse={avg_val:.6f}")

print("Autoencoder training complete ✅")


In [ ]:
#@title Autoencoder loss curves { display-mode: "form" }
# === Autoencoder loss curves ===

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ae_train_losses, label='Train MSE', linewidth=2)
ax.plot(ae_val_losses, label='Val MSE', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE (Reconstruction Error)')
ax.set_title('Approach 2: Autoencoder Training (Real Images Only)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#@title Reconstructions: real vs fake { display-mode: "form" }
# === Visualize: how well does the AE reconstruct real vs fake? ===

autoencoder.eval()
n_show = 6

# Pick some real and fake test images
real_samples = real_test_all[:n_show]
fake_samples = fake_test[:n_show]

with torch.no_grad():
    real_recon = autoencoder(real_samples.to(device)).cpu()
    fake_recon = autoencoder(fake_samples.to(device)).cpu()

fig, axes = plt.subplots(4, n_show, figsize=(2.5 * n_show, 10))
row_labels = ['Real\nOriginal', 'Real\nReconstructed', 'Fake\nOriginal', 'Fake\nReconstructed']

for i in range(n_show):
    axes[0, i].imshow(real_samples[i].permute(1, 2, 0).clamp(0, 1).numpy())
    axes[1, i].imshow(real_recon[i].permute(1, 2, 0).clamp(0, 1).numpy())

    real_err = F.mse_loss(real_recon[i], real_samples[i]).item()
    axes[1, i].set_title(f'MSE={real_err:.4f}', fontsize=8, color='green')

    axes[2, i].imshow(fake_samples[i].permute(1, 2, 0).clamp(0, 1).numpy())
    axes[3, i].imshow(fake_recon[i].permute(1, 2, 0).clamp(0, 1).numpy())

    fake_err = F.mse_loss(fake_recon[i], fake_samples[i]).item()
    axes[3, i].set_title(f'MSE={fake_err:.4f}', fontsize=8, color='red')

for row in range(4):
    for i in range(n_show):
        axes[row, i].axis('off')
    axes[row, 0].set_ylabel(row_labels[row], fontsize=10, rotation=0,
                             labelpad=70, va='center')

plt.suptitle('Autoencoder Reconstruction: Real vs Fake', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
#@title Reconstruction error heatmaps — does this approach stand a chance? { display-mode: "form" }
# DIAGNOSTIC: before we train the probe, let's visually inspect whether the
# reconstruction error actually localizes on fake fruit regions. If the fake
# error maps don't show a visible hot spot over the fruit, no probe on the
# planet can rescue this — we'd need a different approach.

autoencoder.eval()
n_show = 4

with torch.no_grad():
    r = real_test_all[:n_show].to(device)
    f = fake_test[:n_show].to(device)
    r_err = (r - autoencoder(r)).pow(2).mean(dim=1).cpu()   # (n, 128, 128)
    f_err = (f - autoencoder(f)).pow(2).mean(dim=1).cpu()

vmax = max(r_err.max().item(), f_err.max().item())

fig, axes = plt.subplots(4, n_show, figsize=(3 * n_show, 12))
for i in range(n_show):
    axes[0, i].imshow(r[i].cpu().permute(1, 2, 0).numpy())
    axes[1, i].imshow(r_err[i].numpy(), cmap='hot', vmin=0, vmax=vmax)
    axes[2, i].imshow(f[i].cpu().permute(1, 2, 0).numpy())
    axes[3, i].imshow(f_err[i].numpy(), cmap='hot', vmin=0, vmax=vmax)
    for row in range(4):
        axes[row, i].axis('off')

for row, label in enumerate(['Real image', 'Real error', 'Fake image', 'Fake error']):
    axes[row, 0].set_ylabel(label, fontsize=10, rotation=0, labelpad=55, va='center')

plt.suptitle('Reconstruction error maps — this is what the probe sees', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Mean real error: {r_err.mean().item():.6f}")
print(f"Mean fake error: {f_err.mean().item():.6f}")
print(f"Ratio (fake/real): {f_err.mean().item() / max(r_err.mean().item(), 1e-9):.2f}x")


### Step 4 — Stage 2: Reconstruction-Error Probe 🔬

Here's the twist. If we just probe the 128-dim bottleneck embedding, we run into a problem: most of the pixels in every image are **background**, and the diffusion pipeline only repaints the fruit region — leaving the background untouched. Real and fake images share most of their pixels verbatim, so their bottleneck embeddings collapse onto each other and a linear probe can't separate them (AUROC ≈ 0.5, we verified this the hard way 😅).

The fix is to stop looking at *what* the AE encodes and start looking at *where it fails*. We run each image through the frozen AE and compute a per-pixel reconstruction error:

$$\mathrm{err}(x) = \left(x - \mathrm{AE}(x)\right)^2$$

- **Real images** are in-distribution for the AE → it reconstructs them cleanly → the error map is uniformly low.
- **Fake images** have backgrounds the AE has seen a thousand times (reconstructs fine) but an inpainted fruit region that's subtly off-manifold (diffusion artifacts + VAE roundtrip noise the AE was never trained on) → a **hot spot** over the fruit area.

We average-pool the per-pixel error down to an 8×8 grid (64 scalar features per image) and train a tiny `Linear(64 → 1)` probe on those. The probe learns which spatial buckets tend to light up on fakes — **no mask, no localization hints at inference time**. The detector finds its own suspicious regions via the residual. This is textbook anomaly detection wrapped in our frozen-encoder + linear-probe pedagogy.


In [ ]:
#@title Evaluation utilities (metrics, plots, gallery) { display-mode: "form" }
# === Evaluation utilities ===

@torch.no_grad()
def evaluate_binary(labels, probs, title=''):
    """Compute and plot metrics for a binary classifier."""
    preds = (probs > 0.5).astype(int)

    print("=" * 50)
    print(f"CLASSIFICATION REPORT — {title}")
    print("=" * 50)
    print(classification_report(labels, preds,
          target_names=['Real', 'Fake'], digits=3))

    auc_score = roc_auc_score(labels, probs)
    ap = average_precision_score(labels, probs)
    print(f"ROC-AUC: {auc_score:.4f}")
    print(f"Average Precision (PR-AUC): {ap:.4f}")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Confusion Matrix
    cm = confusion_matrix(labels, preds)
    im = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
    axes[0].set_title('Confusion Matrix', fontsize=13)
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')
    axes[0].set_xticks([0, 1])
    axes[0].set_yticks([0, 1])
    axes[0].set_xticklabels(['Real', 'Fake'])
    axes[0].set_yticklabels(['Real', 'Fake'])
    for i in range(2):
        for j in range(2):
            axes[0].text(j, i, str(cm[i, j]),
                        ha='center', va='center', fontsize=16,
                        color='white' if cm[i, j] > cm.max()/2 else 'black')
    fig.colorbar(im, ax=axes[0], fraction=0.046)

    # ROC Curve
    fpr, tpr, _ = roc_curve(labels, probs)
    axes[1].plot(fpr, tpr, linewidth=2, label=f'AUC = {auc_score:.3f}')
    axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('ROC Curve', fontsize=13)
    axes[1].legend(fontsize=12)
    axes[1].grid(True, alpha=0.3)

    # Precision-Recall Curve
    precision, recall, _ = precision_recall_curve(labels, probs)
    axes[2].plot(recall, precision, linewidth=2, label=f'AP = {ap:.3f}')
    axes[2].set_xlabel('Recall')
    axes[2].set_ylabel('Precision')
    axes[2].set_title('Precision-Recall Curve', fontsize=13)
    axes[2].legend(fontsize=12)
    axes[2].grid(True, alpha=0.3)

    plt.suptitle(f'{title} — Test Set Performance', fontsize=15, y=1.02)
    plt.tight_layout()
    plt.show()

    return probs, preds, auc_score, ap


def show_gallery(test_ds, probs, preds, labels, title='', n_per_row=5):
    fake_mask = labels == 1
    real_mask = labels == 0

    tp_idx = np.where((preds == 1) & fake_mask)[0]
    fn_idx = np.where((preds == 0) & fake_mask)[0]
    fp_idx = np.where((preds == 1) & real_mask)[0]

    categories = [
        ('Correctly Detected Fakes (TP)', tp_idx, 'green'),
        ('Missed Fakes (FN)', fn_idx, 'red'),
        ('False Alarms (FP)', fp_idx, 'orange'),
    ]

    fig, axes = plt.subplots(3, n_per_row, figsize=(3 * n_per_row, 9))

    for row, (cat_title, indices, color) in enumerate(categories):
        n_show = min(n_per_row, len(indices))
        if len(indices) > 0:
            sorted_idx = indices[np.argsort(-probs[indices])] if row == 0 else indices[np.argsort(probs[indices])]
        else:
            sorted_idx = indices

        for col in range(n_per_row):
            ax = axes[row, col]
            if col < n_show:
                idx = sorted_idx[col]
                img = test_ds.images[idx]
                ax.imshow(img.permute(1, 2, 0).clamp(0, 1).numpy())
                ax.set_title(f'p={probs[idx]:.2f}', fontsize=9, color=color)
            ax.axis('off')

        axes[row, 0].set_ylabel(f'{cat_title}\n(n={len(indices)})',
                                fontsize=9, rotation=0, labelpad=120, va='center',
                                color=color, fontweight='bold')

    plt.suptitle(f'{title} — Prediction Gallery', fontsize=14)
    plt.tight_layout()
    plt.show()


In [ ]:
# === Stage 2: Reconstruction-error probe on frozen AE ===

class LinearProbe(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.head = nn.Linear(in_dim, 1)

    def forward(self, z):
        return self.head(z)

# Freeze the whole AE — we never update it again
autoencoder.eval()
for p in autoencoder.parameters():
    p.requires_grad = False

# Extract per-image reconstruction-error features.
# We run the image through the frozen AE, square the residual, average across
# color channels, and average-pool the resulting (128x128) error map down to
# an 8x8 grid → 64 scalars per image. Each scalar is the mean squared
# reconstruction error in one spatial bucket.
@torch.no_grad()
def extract_error_features(images, ae, batch_size=64, grid=ERROR_GRID):
    feats = []
    pool_kernel = IMG_SIZE // grid   # 128 // 8 = 16
    for i in range(0, len(images), batch_size):
        x = images[i:i+batch_size].to(device)
        x_hat = ae(x)
        err = (x - x_hat).pow(2).mean(dim=1, keepdim=True)       # (B, 1, 128, 128)
        err_pooled = F.avg_pool2d(err, kernel_size=pool_kernel)  # (B, 1, 8, 8)
        feats.append(err_pooled.flatten(1).cpu())                # (B, 64)
    return torch.cat(feats)

z_real_train = extract_error_features(real_train, autoencoder)
z_fake_train = extract_error_features(fake_train, autoencoder)
z_real_val   = extract_error_features(real_val,   autoencoder)
z_fake_val   = extract_error_features(fake_val,   autoencoder)
z_real_test  = extract_error_features(real_test_all, autoencoder)
z_fake_test  = extract_error_features(fake_test,  autoencoder)

# Normalize features using train-real statistics.
# Without this the probe's decision threshold is pinned to an absolute error
# magnitude — any systematic shift between reals (e.g. test reals reconstructing
# slightly worse than train reals) pushes every test sample above the threshold
# and you end up predicting "fake" for everything. Z-scoring against train-real
# stats makes the probe sensitive to *relative* deviations instead.
feat_mean = z_real_train.mean(dim=0, keepdim=True)
feat_std  = z_real_train.std(dim=0, keepdim=True) + 1e-6

def _norm(z):
    return (z - feat_mean) / feat_std

z_real_train = _norm(z_real_train)
z_fake_train = _norm(z_fake_train)
z_real_val   = _norm(z_real_val)
z_fake_val   = _norm(z_fake_val)
z_real_test  = _norm(z_real_test)
z_fake_test  = _norm(z_fake_test)

FEATURE_DIM = z_real_train.shape[1]
print(f'Feature dim: {FEATURE_DIM}  ({ERROR_GRID}x{ERROR_GRID} spatial buckets)')

probe = LinearProbe(in_dim=FEATURE_DIM).to(device)
print(f'Probe parameters: {sum(p.numel() for p in probe.parameters()):,}')
print(f'AE parameters (frozen): {sum(p.numel() for p in autoencoder.parameters()):,}')

# Balance probe training set: downsample real to match fake count
n_probe = len(z_fake_train)
perm = torch.randperm(len(z_real_train))[:n_probe]
z_real_probe = z_real_train[perm]

probe_z = torch.cat([z_real_probe, z_fake_train])
probe_y = torch.cat([torch.zeros(n_probe), torch.ones(len(z_fake_train))])

# Shuffle
idx = torch.randperm(len(probe_z))
probe_z, probe_y = probe_z[idx], probe_y[idx]

probe_loader = DataLoader(
    torch.utils.data.TensorDataset(probe_z, probe_y),
    batch_size=32, shuffle=True
)

# Val set (keep imbalanced — reflects real evaluation)
val_z = torch.cat([z_real_val, z_fake_val])
val_y = torch.cat([torch.zeros(len(z_real_val)), torch.ones(len(z_fake_val))])

# Train the linear probe
probe_opt = optim.Adam(probe.parameters(), lr=PROBE_LR)
probe_criterion = nn.BCEWithLogitsLoss()

probe_train_losses, probe_val_losses = [], []
for epoch in range(PROBE_EPOCHS):
    probe.train()
    epoch_loss = 0
    for zb, yb in probe_loader:
        zb, yb = zb.to(device), yb.to(device)
        loss = probe_criterion(probe(zb).squeeze(), yb)
        probe_opt.zero_grad()
        loss.backward()
        probe_opt.step()
        epoch_loss += loss.item()
    probe_train_losses.append(epoch_loss / len(probe_loader))

    probe.eval()
    with torch.no_grad():
        val_logits = probe(val_z.to(device)).squeeze()
        probe_val_losses.append(probe_criterion(val_logits, val_y.to(device)).item())

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Probe epoch {epoch+1}/{PROBE_EPOCHS}  train={probe_train_losses[-1]:.4f}  val={probe_val_losses[-1]:.4f}')

print('Reconstruction-error probe training complete ✅')


### Step 5 — Evaluate the Detector 📊

Time for the verdict. We run the frozen-encoder + linear-probe pipeline on the held-out test set (all real test images plus the 25 untouched fakes) and look at ROC-AUC, PR-AUC, the confusion matrix, and a gallery of the probe's most confident catches, its worst misses, and its false alarms.


In [ ]:
# === Evaluate linear probe ===

probe.eval()
test_z = torch.cat([z_real_test, z_fake_test])
test_y = torch.cat([torch.zeros(len(z_real_test)), torch.ones(len(z_fake_test))])

with torch.no_grad():
    test_logits = probe(test_z.to(device)).squeeze().cpu()
    probe_probs_np = torch.sigmoid(test_logits).numpy()
    probe_labels_np = test_y.numpy()

probe_probs_out, probe_preds, probe_auc, probe_ap = evaluate_binary(
    probe_labels_np, probe_probs_np,
    "Approach 2: Frozen Encoder + Linear Probe"
)


In [ ]:
#@title Prediction gallery { display-mode: "form" }
# === Linear probe gallery ===

class SimpleDataset:
    def __init__(self, real_imgs, fake_imgs):
        self.images = torch.cat([real_imgs, fake_imgs], dim=0)

probe_test_ds = SimpleDataset(real_test_all, fake_test)
show_gallery(probe_test_ds, probe_probs_out, probe_preds, probe_labels_np.astype(int),
             "Frozen Encoder + Linear Probe")


## Discussion

### Frozen Encoder + Linear Probe
- **Stage 1**: Autoencoder trained on real images only — MAE-style self-supervised pretext task. The encoder learns a compressed representation of the real data manifold without ever seeing a fake image
- **Stage 2**: Decoder discarded (it served its purpose), encoder frozen. A single linear layer trained on a small balanced set (60 real + 60 fake) finds the decision boundary in the 64-dim latent space
- **Practical advantage**: The encoder is expensive but only needs real data. The probe is trivially cheap — when new deepfake styles emerge, just collect a few examples and retrain the linear head in seconds

### Connection to Course Themes
- **MAE / Representation Learning**: Stage 1 is exactly the MAE paradigm — self-supervised reconstruction as a pretext task to learn useful representations. The quality of the encoder is evaluated by how well a simple linear classifier performs on top
- **Manifolds**: The encoder maps 64x64x3 images (12,288 dims) to a 64-dim latent space. Real images cluster tightly on this learned manifold; fakes are displaced, making them linearly separable
- **Linear Probing**: A standard evaluation protocol for representation quality — if a linear classifier on frozen features works well, the representations have captured meaningful structure
- **Adversarial Arms Race**: A sufficiently good generator could produce fakes that embed near the real cluster — this is the fundamental tension between generators and detectors


### Plan B — Direct Supervised CNN 🏋️

The reconstruction-error probe gave us a taste of what happens when real and fake share most of their pixels: the representation-learning story breaks down because there's nothing distinctive for the AE to specialize on. When the signal is this subtle and this localized, the pragmatic move is to skip the pretraining entirely and train a small CNN end-to-end on real-vs-fake labels.

The tradeoff is honest: we lose the **retrain-a-cheap-head** ergonomic of the two-stage approach — when a new generator shows up we have to retrain the whole classifier, not just a linear probe on top. But diffusion artifacts are notoriously CNN-shaped (characteristic high-frequency signatures from the VAE and the denoising process), and 60 labeled examples stretches much further than you'd think once we turn on augmentation: horizontal/vertical flips, 90° rotations, brightness jitter. Effectively we see thousands of augmented variants over 40 epochs.

This is an honest Week-4 lesson: self-supervised pretraining is a powerful prior when the task leaves room for a generic encoder to shine, but for subtle local artifacts with a tiny label budget, direct supervision wins.


In [ ]:
# === Plan B: Direct supervised CNN with augmentation ===

class FakeDetectorCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),    # 128 -> 64
            nn.GroupNorm(8, 32), nn.GELU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),   # 64 -> 32
            nn.GroupNorm(8, 64), nn.GELU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),  # 32 -> 16
            nn.GroupNorm(8, 128), nn.GELU(),
            nn.Conv2d(128, 128, 3, stride=2, padding=1), # 16 -> 8
            nn.GroupNorm(8, 128), nn.GELU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(128, 1),
        )
    def forward(self, x):
        return self.head(self.features(x))


def augment(x):
    """Per-sample augmentation on a batch. x: (B, 3, H, W) in [0, 1]."""
    out = x.clone()
    for i in range(out.size(0)):
        if random.random() < 0.5:
            out[i] = torch.flip(out[i], dims=[-1])
        if random.random() < 0.5:
            out[i] = torch.flip(out[i], dims=[-2])
        k = random.randint(0, 3)
        if k > 0:
            out[i] = torch.rot90(out[i], k=k, dims=[-2, -1])
        brightness = 1.0 + (random.random() - 0.5) * 0.3
        out[i] = (out[i] * brightness).clamp(0, 1)
    return out


CNN_EPOCHS = 40
CNN_LR = 1e-3
CNN_BATCH = 64

cnn = FakeDetectorCNN().to(device)
print(f"CNN parameters: {sum(p.numel() for p in cnn.parameters()):,}")

# Training set: ALL real_train + ALL fake_train. Balanced batches via
# WeightedRandomSampler — each batch is ~50/50 real/fake, each real is seen
# ~once per epoch, each fake is seen many times per epoch (with different
# augmentation each time). This fixes the failure mode we saw with the
# 60-real balanced setup: too few reals → the classifier never learns a
# robust "what real looks like" and flags unfamiliar test backgrounds as
# fake.
X_train = torch.cat([real_train, fake_train], dim=0)
y_train = torch.cat([torch.zeros(len(real_train)), torch.ones(len(fake_train))], dim=0)

class_counts = torch.tensor([len(real_train), len(fake_train)], dtype=torch.double)
class_weights = 1.0 / class_counts
sample_weights = class_weights[y_train.long()]
sampler = torch.utils.data.WeightedRandomSampler(
    weights=sample_weights,
    num_samples=2 * len(real_train),   # ~1 pass over reals, many over fakes
    replacement=True,
)

cnn_loader = DataLoader(
    torch.utils.data.TensorDataset(X_train, y_train),
    batch_size=CNN_BATCH, sampler=sampler,
)

# Imbalanced val set (matches deployment distribution)
X_val = torch.cat([real_val, fake_val], dim=0)
y_val = torch.cat([torch.zeros(len(real_val)), torch.ones(len(fake_val))], dim=0)

print(f"Training on {len(real_train)} reals + {len(fake_train)} fakes "
      f"(sampler-balanced, {len(cnn_loader)} batches/epoch)")

cnn_opt = optim.Adam(cnn.parameters(), lr=CNN_LR, weight_decay=1e-4)
cnn_criterion = nn.BCEWithLogitsLoss()

for epoch in range(CNN_EPOCHS):
    cnn.train()
    epoch_loss = 0.0
    for xb, yb in cnn_loader:
        xb = augment(xb.to(device))
        yb = yb.to(device)
        logits = cnn(xb).squeeze()
        loss = cnn_criterion(logits, yb)
        cnn_opt.zero_grad()
        loss.backward()
        cnn_opt.step()
        epoch_loss += loss.item()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        cnn.eval()
        with torch.no_grad():
            val_logits = cnn(X_val.to(device)).squeeze()
            val_auc = roc_auc_score(y_val.numpy(), torch.sigmoid(val_logits).cpu().numpy())
        print(f"Epoch {epoch+1:3d}/{CNN_EPOCHS}  train_loss={epoch_loss/len(cnn_loader):.4f}  val_auc={val_auc:.4f}")

print("CNN training complete ✅")

# === Evaluate on test set ===
cnn.eval()
X_test = torch.cat([real_test_all, fake_test], dim=0)
y_test = torch.cat([torch.zeros(len(real_test_all)), torch.ones(len(fake_test))], dim=0)

with torch.no_grad():
    test_logits = []
    for i in range(0, len(X_test), 64):
        test_logits.append(cnn(X_test[i:i+64].to(device)).squeeze().cpu())
    test_logits = torch.cat(test_logits)
    cnn_probs = torch.sigmoid(test_logits).numpy()
    cnn_labels = y_test.numpy()

cnn_probs_out, cnn_preds, cnn_auc, cnn_ap = evaluate_binary(
    cnn_labels, cnn_probs, "Plan B: Supervised CNN + Augmentation"
)

# Gallery
cnn_test_ds = SimpleDataset(real_test_all, fake_test)
show_gallery(cnn_test_ds, cnn_probs_out, cnn_preds, cnn_labels.astype(int),
             "Supervised CNN + Augmentation")


### Plan C — Anomaly Detection with Pretrained Features 🎯

Plan A (reconstruction-error probe) and Plan B (supervised CNN) both hit the same wall: real and fake images share ~80% of their pixels verbatim — the diffusion pipeline only repaints the fruit region. The discriminating signal is a small, localized, *semantic* swap (apple → orange), and neither our from-scratch autoencoder nor a 60-fake-trained CNN has a feature space that treats "apple vs. orange" as a first-class axis of variation.

But **ImageNet does**. A ResNet50 pretrained on ImageNet-1k was forced to distinguish apples and oranges as separate classes during training, so its penultimate features place them in different regions of a 2048-dim manifold. We don't need to train anything — just borrow those features.

This also reframes the task honestly. With ~2400 real images and only 60 fakes, this is a **one-class / anomaly detection** problem, not a supervised classification problem. The right move is:

1. Fit a model of "real" on all 2400 real-train images.
2. Score each test image by its distance from the real-train manifold.
3. Reserve the fake labels for evaluation only — we never fit on them.

Concretely:

- Run every image through a frozen ImageNet ResNet50 → one 2048-dim feature vector per image.
- Project to 128-d via PCA on `real_train` features (this decorrelates the covariance and keeps Mahalanobis well-conditioned — raw 2048-d covariance from 2400 samples would be rank-deficient).
- In PCA space, compute the diagonal Mahalanobis distance of each test image from the `real_train` mean.
- Higher distance → more off-manifold → more likely fake.

The calibration trick from Plan A stays: z-score the test distances against `real_train` distances before sigmoid, so `evaluate_binary`'s default 0.5 threshold actually lives near the real/fake boundary instead of collapsing to "all fake".


In [ ]:
# === Plan C: Pretrained ResNet50 + Mahalanobis anomaly detector ===
from torchvision.models import resnet50, ResNet50_Weights

# ---- Frozen feature extractor (ResNet50, ImageNet, avgpool output) ----
_resnet = resnet50(weights=ResNet50_Weights.DEFAULT).to(device).eval()
for p in _resnet.parameters():
    p.requires_grad = False
_resnet.fc = nn.Identity()  # forward(x) -> (B, 2048)

_IM_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
_IM_STD  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

@torch.no_grad()
def extract_resnet_features(images, batch_size=32):
    """Resize 128->224, ImageNet-normalize, extract (N, 2048) avgpool features on CPU."""
    feats = []
    for i in range(0, len(images), batch_size):
        x = images[i:i+batch_size].to(device)
        x = F.interpolate(x, size=224, mode='bilinear', align_corners=False)
        x = (x - _IM_MEAN) / _IM_STD
        feats.append(_resnet(x).cpu())
    return torch.cat(feats)

print("Extracting ResNet50 features...")
f_real_train = extract_resnet_features(real_train)
f_real_test  = extract_resnet_features(real_test_all)
f_fake_test  = extract_resnet_features(fake_test)
print(f"  real_train: {tuple(f_real_train.shape)}")
print(f"  real_test:  {tuple(f_real_test.shape)}")
print(f"  fake_test:  {tuple(f_fake_test.shape)}")

# ---- Fit Mahalanobis in PCA-projected space (real_train only, no fakes) ----
PCA_DIM = 128
torch.manual_seed(0)

mean_rt = f_real_train.mean(dim=0, keepdim=True)              # (1, 2048)
centered = f_real_train - mean_rt
_, _, V = torch.pca_lowrank(centered, q=PCA_DIM, center=False)
V_k = V[:, :PCA_DIM]                                           # (2048, 128)

def _project(feats):
    return (feats - mean_rt) @ V_k                            # (N, 128)

proj_train = _project(f_real_train)
inv_var = 1.0 / (proj_train.var(dim=0, unbiased=False) + 1e-6)  # (128,) diagonal cov in PCA space

def mahalanobis(feats):
    p = _project(feats)
    return (p * p * inv_var).sum(dim=1)                       # squared distance from train mean

# ---- Calibrate against train-real distances (not test) ----
d_train = mahalanobis(f_real_train)
d_mu, d_sd = d_train.mean(), d_train.std() + 1e-6

def to_prob(d):
    return torch.sigmoid((d - d_mu) / d_sd).numpy()

d_real_test = mahalanobis(f_real_test)
d_fake_test = mahalanobis(f_fake_test)
print(f"Mean Mahalanobis  real_test={d_real_test.mean().item():.2f}  "
      f"fake_test={d_fake_test.mean().item():.2f}  "
      f"(ratio {d_fake_test.mean().item() / max(d_real_test.mean().item(), 1e-9):.2f}x)")

plan_c_probs = np.concatenate([to_prob(d_real_test), to_prob(d_fake_test)])
plan_c_labels = np.concatenate([np.zeros(len(f_real_test)),
                                np.ones(len(f_fake_test))]).astype(int)

plan_c_probs_out, plan_c_preds, plan_c_auc, plan_c_ap = evaluate_binary(
    plan_c_labels, plan_c_probs, "Plan C: ResNet50 + Mahalanobis"
)

# Gallery
plan_c_test_ds = SimpleDataset(real_test_all, fake_test)
show_gallery(plan_c_test_ds, plan_c_probs_out, plan_c_preds, plan_c_labels,
             "Plan C: ResNet50 + Mahalanobis")


### Diagnostic — GMM on AE latents 🔬

Before committing to ResNet50 features, let's test a cheaper hypothesis: can we detect fakes using a **Gaussian Mixture Model** on the latents of the autoencoder we already trained?

The logic: Mahalanobis = single Gaussian log-likelihood. GMM is the multi-modal upgrade — it fits several Gaussians to capture different "real image clusters" (different backgrounds, lighting, compositions). If the real distribution is multi-modal in latent space, a GMM will give a much tighter fit than a single Gaussian, and fakes that sit in low-density pockets will get flagged.

**But here's the hypothesis I'm actually testing:** I suspect the AE is *blind* to the apple→orange swap. Pixel-MSE training on images that share ~80% of background pixels forces the 128-d bottleneck to spend its capacity on encoding backgrounds, not on semantic fruit identity. If that's true, fakes will sit *on* the real manifold in latent space — and no density estimator (GMM, KDE, normalizing flow) can separate points that are co-located.

**Success criterion:** `fake_nll / real_nll > 1.5x`. If we get ~1.0, the hypothesis is confirmed and we move on to Plan C (pretrained features). If we get >1.5x, the AE features have more signal than I thought and it's worth running a full GMM-based probe.


In [ ]:
# === Diagnostic: GMM on AE latents ===
from sklearn.mixture import GaussianMixture

autoencoder.eval()

@torch.no_grad()
def encode_all(images, batch_size=128):
    zs = []
    for i in range(0, len(images), batch_size):
        x = images[i:i+batch_size].to(device)
        zs.append(autoencoder.encode(x).cpu())
    return torch.cat(zs).numpy()

print("Encoding through AE...")
z_real_train_np = encode_all(real_train)
z_real_test_np  = encode_all(real_test_all)
z_fake_test_np  = encode_all(fake_test)
print(f"  real_train: {z_real_train_np.shape}")
print(f"  real_test:  {z_real_test_np.shape}")
print(f"  fake_test:  {z_fake_test_np.shape}")

# Fit GMM on real_train latents only (no fakes touched)
N_COMPONENTS = 8
print(f"Fitting GMM (n_components={N_COMPONENTS}, covariance_type='diag')...")
gmm = GaussianMixture(
    n_components=N_COMPONENTS,
    covariance_type='diag',
    random_state=0,
    reg_covar=1e-4,
    max_iter=200,
).fit(z_real_train_np)
print(f"  converged: {gmm.converged_}  lower_bound: {gmm.lower_bound_:.2f}")

# Score samples: GMM returns log-likelihood; we want negative log-likelihood
# as an anomaly score (higher = more anomalous).
nll_real_train = -gmm.score_samples(z_real_train_np)
nll_real_test  = -gmm.score_samples(z_real_test_np)
nll_fake_test  = -gmm.score_samples(z_fake_test_np)

print()
print("=== Anomaly score (negative log-likelihood) ===")
print(f"  real_train:  mean={nll_real_train.mean():.2f}  std={nll_real_train.std():.2f}")
print(f"  real_test:   mean={nll_real_test.mean():.2f}  std={nll_real_test.std():.2f}")
print(f"  fake_test:   mean={nll_fake_test.mean():.2f}  std={nll_fake_test.std():.2f}")

ratio = nll_fake_test.mean() / max(abs(nll_real_test.mean()), 1e-9)
print(f"  ratio (fake/real_test): {ratio:.2f}x")
print()
if ratio > 1.5:
    print("✅ Signal detected — worth running a full GMM probe on these features.")
elif ratio > 1.1:
    print("⚠️  Marginal signal. Might work with careful calibration, but Plan C likely stronger.")
else:
    print("❌ AE latents are blind — confirms hypothesis. Proceed with Plan C (ResNet50 features).")

# Quick AUROC check — the honest metric
from sklearn.metrics import roc_auc_score
y = np.concatenate([np.zeros(len(nll_real_test)), np.ones(len(nll_fake_test))])
scores = np.concatenate([nll_real_test, nll_fake_test])
auroc = roc_auc_score(y, scores)
print(f"  AUROC (GMM on AE latents): {auroc:.4f}")

# Visualize score distributions
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.hist(nll_real_test, bins=40, alpha=0.6, label='real_test', color='steelblue', density=True)
ax.hist(nll_fake_test, bins=40, alpha=0.6, label='fake_test', color='crimson', density=True)
ax.set_xlabel('Negative log-likelihood under GMM')
ax.set_ylabel('Density')
ax.set_title(f'GMM on AE latents — AUROC {auroc:.3f}, ratio {ratio:.2f}x')
ax.legend()
plt.tight_layout()
plt.show()
